In [ ]:
import re
import os

from pathlib import Path


def extract_parameters_from_filename(filename: str):
    """Extracts the x, y coordinates and the radius from the filename."""
    numbers = re.findall(r"\d+", filename)

    if len(numbers) >= 3:
        radius = numbers[-1]
        y_coordinate = numbers[-2]
        x_coordinate = numbers[-3]

        return int(x_coordinate), int(y_coordinate), int(radius)
    else:
        raise ValueError("The filename does not contain enough numbers.")


def get_original_stem(stem: str):
    return "_".join(stem.split("_")[0:-3])


gen_images_path = Path("../gen_data/patched_images")

files = [file for file in gen_images_path.iterdir() if file.is_file()]

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import matplotlib.pyplot as plt
import shutil
import cv2
import numpy as np


# Function that gets executed based on user's button click
def process_image(image_path: Path, decision: str):
    shutil.move(
        str(image_path),
        str(image_path.parent.parent / decision / image_path.name),
    )


# Function to display each image and wait for user input
def display_image(patch_image_path: Path):
    original_image_stem = get_original_stem(patch_image_path.stem)
    print(f"Original image: {original_image_stem}")

    x, y, radius = extract_parameters_from_filename(patch_image_path.name)

    # Cut out circle patch from image using opencv
    img_cv2 = cv2.imread(str(patch_image_path))

    img_cv22 = img_cv2.copy()
    cv2.circle(img_cv22, (x, y), radius, (0, 0, 255), 10)

    mask = np.zeros(img_cv2.shape[:2], np.uint8)
    mask = cv2.circle(mask, (x, y), radius, (255, 255, 255), -1)
    img_cv2 = cv2.bitwise_and(img_cv2, img_cv2, mask=mask)

    # Crop black parts from cropped image
    img_cv2 = img_cv2[
        np.min(np.where(img_cv2 != 0)[0]) : np.max(np.where(img_cv2 != 0)[0]),
        np.min(np.where(img_cv2 != 0)[1]) : np.max(np.where(img_cv2 != 0)[1]),
    ]

    img_cv22 = cv2.cvtColor(img_cv22, cv2.COLOR_BGR2RGB)
    img_cv2 = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)

    # Display the image
    axs, fig = plt.subplots(1, 2, figsize=(10, 5))
    fig[0].imshow(img_cv22)
    fig[0].axis("off")
    fig[1].imshow(img_cv2)
    fig[1].axis("off")
    plt.tight_layout()
    plt.show()

    # Create Yes and No buttons
    btn_yes = widgets.Button(description="Yes")
    btn_no = widgets.Button(description="No")

    # Define button click actions
    def on_yes_clicked(b):
        clear_output()
        process_image(patch_image_path, "Yes")
        display_next_image()

    def on_no_clicked(b):
        clear_output()
        process_image(patch_image_path, "No")
        display_next_image()

    btn_yes.on_click(on_yes_clicked)
    btn_no.on_click(on_no_clicked)

    # Display buttons
    display(widgets.HBox([btn_yes, btn_no]))


# Function to control the display of images
def display_next_image():
    if not files:  # If no more images are left
        print("No more images to display.")
        return
    next_image = files.pop(0)  # Get the next image
    display_image(next_image)


# Start the image display
display_next_image()